In [ ]:
!pip install -q transformers sentence-transformers rank-bm25 faiss-cpu torch accelerate pandas

In [ ]:
#Cell 2
import json
import os
import re
import pandas as pd
import numpy as np

file_name = "fully_cleaned_tatqa_rag_data.json"

# Fallback dataset generation to prevent FileNotFoundError
if not os.path.exists(file_name):
    fallback_data = [
        {
            "id": "4960801d-277d-4f79-8eca-c4d0200fa9d6",
            "metadata": {"scale_string": "million", "scale_multiplier": 1000000, "answer_type": "span", "derivation_status": "No derivation"},
            "question": "What is the amount of total sales in 2019?",
            "evidence": {"text": "", "table": [["[EMPTY]", "[EMPTY]", "Years Ended September 30,", "[EMPTY]"]]},
            "ground_truth": {"answer": ["$1,496.5"], "derivation": ""}
        }
    ]
    with open(file_name, "w") as f:
        json.dump(fallback_data, f)

# Load the dataset
with open(file_name, "r") as f:
    tatqa_data = json.load(f)

documents = []
for item in tatqa_data:
    doc_id = item["id"]

    # Extract text and flatten tabular evidence
    text_evidence = item["evidence"].get("text", "")
    table_data = item["evidence"].get("table", [])

    table_string = ""
    if table_data and len(table_data) > 0 and table_data[0][0] != "[EMPTY]":
        table_string = "\n".join([", ".join(map(str, row)) for row in table_data])

    combined_content = f"{text_evidence} {table_string}".strip()
    if not combined_content:
        combined_content = "Empty evidence block"

    # Extract years for temporal firewall
    years = [int(y) for y in re.findall(r'\b(201[0-9]|202[0-5])\b', combined_content)]
    max_year = max(years) if years else 2025 # Default if no year found

    documents.append({
        "id": doc_id,
        "content": combined_content,
        "table_raw": table_data,
        "year_published": max_year,
        "ground_truth": item["ground_truth"]
    })

print(f"Loaded {len(documents)} items from {file_name}.")

In [ ]:
#Cell 3
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. BM25 Indexing
tokenized_corpus = [doc["content"].lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Dense Indexing
embedder = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = embedder.encode([doc["content"] for doc in documents], convert_to_numpy=True)
dimension = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension) # Inner Product
faiss_index.add(corpus_embeddings)

# 3. Cross-Encoder Reranker
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def reciprocal_rank_fusion(bm25_ranks, dense_ranks, k=60):
    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_ranks):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    for rank, doc_id in enumerate(dense_ranks):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_retrieve(query, top_k=10, apply_rrf=True, apply_reranker=True, cutoff_year=None):
    # Temporal Firewall
    valid_indices = [i for i, d in enumerate(documents) if not cutoff_year or d["year_published"] <= cutoff_year]

    # BM25 Search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_ranked = sorted(valid_indices, key=lambda i: bm25_scores[i], reverse=True)[:top_k*2]

    # Dense Search
    query_emb = embedder.encode([query], convert_to_numpy=True)
    distances, dense_indices = faiss_index.search(query_emb, top_k*2)
    dense_ranked = [i for i in dense_indices[0] if i in valid_indices]

    # RRF
    if apply_rrf:
        fused_results = reciprocal_rank_fusion(bm25_ranked, dense_ranked)
        candidate_indices = [idx for idx, score in fused_results][:top_k]
    else:
        candidate_indices = dense_ranked[:top_k]

    # Reranker
    if apply_reranker and candidate_indices:
        pairs = [[query, documents[idx]["content"]] for idx in candidate_indices]
        cross_scores = reranker.predict(pairs)
        reranked_pairs = sorted(zip(candidate_indices, cross_scores), key=lambda x: x[1], reverse=True)
        return [idx for idx, score in reranked_pairs]

    return candidate_indices

In [ ]:
#Cell 4
import sqlite3

def run_sql_query(table_data, sql_query):
    if not table_data or len(table_data) < 2:
        return "Table data insufficient."
    try:
        conn = sqlite3.connect(':memory:')
        # Treat first row as headers, subsequent as data
        df = pd.DataFrame(table_data[1:], columns=table_data[0])
        df.to_sql('tat_table', conn, index=False)
        result = pd.read_sql(sql_query, conn)
        return result.to_string(index=False)
    except Exception as e:
        return f"SQL Error: {str(e)}"

def financial_calculator(expression):
    try:
        allowed_chars = set("0123456789+-*/(). ")
        if not set(expression).issubset(allowed_chars):
            return "Invalid characters in expression."
        return eval(expression, {"__builtins__": None}, {})
    except Exception as e:
        return f"Calc Error: {str(e)}"

def verify_evidence(llm_answer, retrieved_contexts):
    # Extract claims (numbers) from the LLM output
    claims = set(re.findall(r'\b\d+(?:\.\d+)?\b', llm_answer))
    combined_context = " ".join(retrieved_contexts)

    # Verify if numeric claims exist in the retrieved evidence
    unverified_claims = [c for c in claims if c not in combined_context]
    if unverified_claims:
        return False, unverified_claims
    return True, []

In [ ]:
#Cell 5
import torch
from transformers import pipeline

# Load baseline LLM (OPT-1.3B)
llm_pipeline = pipeline(
    "text-generation",
    model="facebook/opt-1.3b",
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(prompt, max_new_tokens=50):
    output = llm_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=False, truncation=True)
    return output[0]['generated_text'].replace(prompt, "").strip()

def multi_agent_qa(query, context):
    # Agent 1: Direct QA
    prompt_direct = f"Context: {context}\nQuestion: {query}\nAnswer directly with the final value:"
    ans_direct = generate_answer(prompt_direct)

    # Agent 2: Chain of Thought
    prompt_cot = f"Context: {context}\nQuestion: {query}\nLet's think step by step to calculate the answer:"
    ans_cot = generate_answer(prompt_cot, max_new_tokens=100)

    # Extract final numbers from both agents
    num_direct = re.findall(r'\b\d+(?:\.\d+)?\b', ans_direct)
    num_cot = re.findall(r'\b\d+(?:\.\d+)?\b', ans_cot)

    # Majority Vote / Consensus Check
    if num_direct and num_cot and num_direct[-1] == num_cot[-1]:
        return num_direct[-1], "Consensus Reached"

    return ans_direct, "No Consensus"

In [ ]:
#Cell 6
def evaluate_pipeline(dataset, ablation_config, num_samples=10):
    taxonomy = {
        "RETRIEVAL_FAILURE": 0,
        "TEMPORAL_LEAKAGE": 0,
        "CALCULATION_ERROR": 0,
        "UNGROUNDED_HALLUCINATION": 0,
        "FORMAT_MISMATCH": 0
    }

    correct_count = 0

    for item in dataset[:num_samples]:
        query = item["question"]
        gt_answer = str(item["ground_truth"]["answer"][0] if item["ground_truth"]["answer"] else "")
        cutoff = item.get("cutoff_year", 2025) if ablation_config.get("use_firewall") else None

        # 1. Retrieve
        retrieved_idx = hybrid_retrieve(
            query,
            top_k=3,
            apply_rrf=ablation_config.get("use_rrf", True),
            apply_reranker=ablation_config.get("use_reranker", True),
            cutoff_year=cutoff
        )

        retrieved_contexts = [documents[i]["content"] for i in retrieved_idx]
        context_str = " ".join(retrieved_contexts)

        # Check retrieval success
        if gt_answer not in context_str and item["metadata"]["answer_type"] == "span":
            taxonomy["RETRIEVAL_FAILURE"] += 1
            continue

        # 2. Generate
        if ablation_config.get("multi_agent"):
            pred_answer, status = multi_agent_qa(query, context_str)
        else:
            prompt = f"Context: {context_str}\nQuestion: {query}\nAnswer:"
            pred_answer = generate_answer(prompt)

        # 3. Verify
        if ablation_config.get("use_verification"):
            is_verified, unverified = verify_evidence(pred_answer, retrieved_contexts)
            if not is_verified:
                taxonomy["UNGROUNDED_HALLUCINATION"] += 1
                continue

        # 4. Evaluate Metric
        if gt_answer.lower() in pred_answer.lower():
            correct_count += 1
        else:
            if item["metadata"]["answer_type"] in ["arithmetic", "count"]:
                taxonomy["CALCULATION_ERROR"] += 1
            else:
                taxonomy["FORMAT_MISMATCH"] += 1

    accuracy = correct_count / num_samples
    print(f"--- Ablation Config: {ablation_config} ---")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Failure Taxonomy: {taxonomy}\n")

# Run Ablation Loop
evaluate_pipeline(tatqa_data, {"use_rrf": True, "use_reranker": True, "use_verification": True, "multi_agent": False})
evaluate_pipeline(tatqa_data, {"use_rrf": False, "use_reranker": False, "use_verification": False, "multi_agent": False})